# Independent CPU readback of saved Development group02 compact proofs
Reopens exact saved proof bytes for shards 0002 and 0003; no training or Final.

In [ ]:
import hashlib, json, pathlib

SOURCE_NOTEBOOK = "hungluongduc/hung-compact-proof-group02-20261028-v1"
SOURCE_VERSION_ID = 363105577
PROOFS = {
    "shard_0002": {
        "bytes": 21388,
        "sha256": "735a225b2c6253b3357c325ae09735110fe39956c17e0dbbec037e064218a99b",
        "archive_bytes": 61939216,
        "archive_sha256": "409c523b421397fecc22bc953a13b2642aa49cf1f0f443c9e88f1d11af4ca908",
        "jobset_sha256": "1cf89024e9c8b9d8488b386a54a0995924fad19a9334f9539d50843d345260f1",
    },
    "shard_0003": {
        "bytes": 21346,
        "sha256": "e23351b9593f1703a075e9fb2ddc935d35f52c43469dbd1b3c59db32fab7e5d8",
        "archive_bytes": 62632926,
        "archive_sha256": "f7f5073878345a4cb44f1e293a7bfd1e1c762244475cf10e7bba40167c66430b",
        "jobset_sha256": "145d75ce5a847f1494d7ec4285e47812c769acb7206bc0da7a20a48e4545f4d8",
    },
}
source_root = pathlib.Path("/kaggle/input")
seen_jobs = set()
readbacks = []
for shard_id, pin in PROOFS.items():
    matches = [
        path for path in source_root.rglob(f"{shard_id}-compact-proof.json")
        if "hung-compact-proof-group02-20261028-v1" in str(path)
    ]
    assert len(matches) == 1, f"Saved proof missing or ambiguous: {shard_id}"
    raw = matches[0].read_bytes()
    assert len(raw) == pin["bytes"]
    assert hashlib.sha256(raw).hexdigest() == pin["sha256"]
    proof = json.loads(raw)
    assert proof["schema"] == "hung-kaggle-compact-shard-proof-v1"
    assert proof["scope"] == "development_only"
    assert proof["shard_id"] == shard_id
    assert proof["archive_sha256"] == pin["archive_sha256"]
    assert proof["archive_bytes"] == pin["archive_bytes"]
    rows = proof["rows"]
    ids = [row["job_id"] for row in rows]
    assert len(ids) == len(set(ids)) == 20
    assert all(row["status"] == "VERIFIED_ARCHIVE_BYTES" and row["shard_id"] == shard_id for row in rows)
    assert not seen_jobs.intersection(ids)
    seen_jobs.update(ids)
    jobset_sha = hashlib.sha256(json.dumps(ids, separators=(",", ":")).encode("utf-8")).hexdigest()
    assert jobset_sha == pin["jobset_sha256"]
    readbacks.append({
        "shard_id": shard_id, "proof_bytes": len(raw),
        "proof_sha256": hashlib.sha256(raw).hexdigest(),
        "jobset_sha256": jobset_sha, "job_ids": ids,
        "jobs": 20, "status": "REMOTE_PROOF_READBACK_VERIFIED",
    })
assert len(seen_jobs) == 40
readback = {
    "schema": "hung-kaggle-compact-proof-group-readback-v1",
    "scope": "development_only",
    "source_notebook": SOURCE_NOTEBOOK,
    "claimed_source_version_id": SOURCE_VERSION_ID,
    "readbacks": readbacks,
    "jobs": 40,
    "status": "REMOTE_PROOF_READBACK_VERIFIED",
}
out = pathlib.Path("/kaggle/working/hung-compact-proof-group02-readback-20261028-v1.json")
with out.open("x", encoding="utf-8", newline="\n") as stream:
    json.dump(readback, stream, sort_keys=True, indent=2, allow_nan=False)
    stream.write("\n")
print(json.dumps({"status": readback["status"], "jobs": 40, "readbacks": [
    {"shard_id": row["shard_id"], "proof_bytes": row["proof_bytes"],
     "proof_sha256": row["proof_sha256"], "jobset_sha256": row["jobset_sha256"]}
    for row in readbacks
]}, sort_keys=True))
